In [1]:
import pandas as pd, numpy as np
import vivarium_inputs
import gbd_mapping
import pathlib

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "India"

In [3]:
# Parameters
location = "india"


In [4]:
location = location.title()

In [5]:
pop = vivarium_inputs.get_population_structure(location).value
pop[pop > 0]

location  sex     age_start  age_end     year_start  year_end
India     Female  0.000000   0.019178    2021        2022        1.975581e+05
                  0.019178   0.076712    2021        2022        5.872100e+05
                  0.076712   0.500000    2021        2022        4.326738e+06
                  0.500000   1.000000    2021        2022        5.085507e+06
                  1.000000   2.000000    2021        2022        1.035319e+07
                  2.000000   5.000000    2021        2022        3.244181e+07
                  5.000000   10.000000   2021        2022        5.860247e+07
                  10.000000  15.000000   2021        2022        6.308955e+07
                  15.000000  20.000000   2021        2022        6.421417e+07
                  20.000000  25.000000   2021        2022        6.367111e+07
                  25.000000  30.000000   2021        2022        5.987495e+07
                  30.000000  35.000000   2021        2022        5.583203e+07
  

In [6]:
children = pop[pop.index.get_level_values("age_end") <= 5].sum()
f'{int(children):,}'

'111,336,281'

In [7]:
pd.read_parquet(f"../../0200_pregnancy_sim/sim_results/{location.lower()}/pregnancy_outcome_count.parquet")

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,lowest,intervention,1,0,0.0
1,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,second,intervention,1,0,0.0
2,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,middle,intervention,1,0,0.0
3,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,fourth,intervention,1,0,0.0
4,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,highest,intervention,1,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
53995,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,stillbirth,95_plus,severe,lowest,baseline,11,0,0.0
53996,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,stillbirth,95_plus,severe,second,baseline,11,0,0.0
53997,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,stillbirth,95_plus,severe,middle,baseline,11,0,0.0
53998,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,stillbirth,95_plus,severe,fourth,baseline,11,0,0.0


In [8]:
sim_baseline_children = pd.read_parquet(f"../../0200_pregnancy_sim/sim_results/{location.lower()}/pregnancy_outcome_count.parquet")
sim_baseline_children = sim_baseline_children[
    (sim_baseline_children.scenario == 'baseline') &
    (sim_baseline_children.sub_entity == 'live_birth')
]
sim_baseline_children

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
2705,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,lowest,baseline,10,0,5.0
2706,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,second,baseline,10,0,3.0
2707,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,middle,baseline,10,0,4.0
2708,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,fourth,baseline,10,0,4.0
2709,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,highest,baseline,10,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
53990,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,lowest,baseline,11,0,0.0
53991,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,second,baseline,11,0,0.0
53992,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,middle,baseline,11,0,0.0
53993,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,fourth,baseline,11,0,0.0


In [9]:
sim_baseline_children = sim_baseline_children.groupby("input_draw").value.sum().mean()
sim_baseline_children

288766.0

In [10]:
scalar = children / sim_baseline_children
scalar

385.5588320220137

In [11]:
for result in ["ylds", "ylls", "deaths"]:
    df = pd.read_parquet(f"../../0300_child_sim/sim_results/{location.lower()}/{result}.parquet")
    df.value *= scalar
    path = pathlib.Path(f'./child_results/{location.lower()}/{result}.parquet')
    path.parent.mkdir(exist_ok=True, parents=True)
    df.to_parquet(path)